# EcoSort model messing around 🗑️( For shivam )

> **Not used by the project.** The real training code is `Brain.py`.

This is my scratch notebook from trying ideas for the waste model. Some notes are unfinished and some cells are old experiments. Keeping it like this so I remember the process.

- [x] Check CUDA
- [x] Try EfficientNet
- [ ] Take more actual webcam photos
- [ ] Try MobileNet for Pi later


In [1]:
# just checking before I start
from pathlib import Path
import torch
print('cuda?', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))
DATA = Path('data') # maybe wrong if notebook opens from parent folder


cuda? True
NVIDIA GeForce RTX 4060 Laptop GPU


## First issue

At first this crashed because I had no dataset. Leaving this failed attempt here.


In [2]:
# from torchvision.datasets import ImageFolder
# ds = ImageFolder('data')  # failed before images were downloaded
# print(len(ds))


FileNotFoundError: old run: Dataset folder not found: data

In [3]:
# not elegant but wanted to check the folders
wanted = ['biodegradable', 'non_biodegradable', 'recyclable']
for name in wanted:
    folder = DATA / name
    images = list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png'))
    print(name, '->', len(images), 'images')
# TODO: some files are webp? check later


## Random image check

Wanted to make sure the downloaded data was actually waste and not broken files.


In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image
all_imgs = []
for label in wanted:
    all_imgs += [(label, p) for p in (DATA / label).glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]
label, p = random.choice(all_imgs)
plt.imshow(Image.open(p).convert('RGB'))
plt.title(label + ' / ' + p.name)
plt.axis('off');


## Old model attempt

Started with EfficientNet from a tutorial. It was okay but the proper model is now ConvNeXt-Tiny in `Brain.py`.


In [4]:
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from torch import nn
print('trying efficientnet... (not final model)')
tmp_model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
tmp_model.classifier[1] = nn.Linear(tmp_model.classifier[1].in_features, 3)
# proper split/checkpoint stuff got moved to Brain.py


trying efficientnet... (not final model)


In [ ]:
# rough idea only -- do not use this to train
# optimizer = torch.optim.Adam(tmp_model.parameters(), lr=0.0001)
# for epoch in range(10):
#     ...

# dirty plastic bottle != clean recyclable bottle
# need more webcam images before saying accuracy is good


## Camera notes

The camera test is in `camera_test.py`. If confidence is low, don't move the servo. 70% is just a starting guess.


In [ ]:
# commands I kept forgetting:
# python Brain.py train --data-dir data --epochs 20 --batch-size 32
# python camera_test.py --checkpoint outputs/ecosort_convnext_tiny_best.pt
confidence_threshold = 0.70
print('below', confidence_threshold, '= retry image / do not sort')


---

This notebook is intentionally only a model diary/testing area. Use `Brain.py` for real training and `camera_test.py` for webcam testing.
